# Qwen3 Voice Cloning - Self-Testing Colab Notebook

This notebook performs voice cloning using Qwen3-TTS with automatic validation, retry logic, and error handling.

**Reference Audio:** https://archive.org/download/ppiirqpwrygv7ehggjvlaxcuz6fn8_w/pPiiRQpWrygV7ehgGjvl%2BaXcuz6fn8_w.mp3

In [ ]:
# CELL 1: Environment Setup & GPU Detection
import os, sys, time, logging
from pathlib import Path
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def validate_environment():
    try:
        is_colab = 'COLAB_GPU' in os.environ
        logger.info(f'Running in Colab: {is_colab}')
        import subprocess
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
        if result.returncode == 0:
            logger.info('GPU detected and available')
        else:
            logger.warning('No GPU detected, will use CPU')
        import torch
        cuda_available = torch.cuda.is_available()
        logger.info(f'CUDA Available: {cuda_available}')
        if cuda_available:
            logger.info(f'GPU Device: {torch.cuda.get_device_name(0)}')
        return True
    except Exception as e:
        logger.error(f'Environment validation failed: {e}')
        return False

success = validate_environment()
assert success, 'Environment validation failed'
print('Cell executed successfully')

In [ ]:
# CELL 2: Package Installation with Retry Logic
import subprocess, sys, time

def install_package(package, max_retries=3):
    for attempt in range(max_retries):
        try:
            logger.info(f'Installing {package} (attempt {attempt + 1}/{max_retries})')
            result = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', package], capture_output=True, text=True, timeout=300)
            if result.returncode == 0:
                logger.info(f'Successfully installed {package}')
                return True
        except Exception as e:
            logger.warning(f'Installation error: {e}')
        if attempt < max_retries - 1:
            time.sleep(5)
    return False

def install_ffmpeg():
    try:
        result = subprocess.run(['which', 'ffmpeg'], capture_output=True, text=True)
        if result.returncode != 0:
            logger.info('Installing ffmpeg...')
            subprocess.run(['apt-get', 'update', '-qq'], check=True, capture_output=True)
            subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True, capture_output=True)
            logger.info('ffmpeg installed')
        else:
            logger.info('ffmpeg already installed')
        return True
    except Exception as e:
        logger.error(f'ffmpeg installation failed: {e}')
        return False

packages = ['torch', 'torchaudio', 'transformers>=4.40.0', 'accelerate', 'soundfile', 'librosa', 'numpy', 'requests', 'IPython']
all_installed = all(install_package(pkg) for pkg in packages)
install_ffmpeg()
try:
    import torch
    logger.info(f'Torch version: {torch.__version__}')
except ImportError:
    logger.warning('Torch not available, may need runtime restart')
print('Cell executed successfully')

In [ ]:
# CELL 3: Import Libraries with Validation
import torch, torchaudio, numpy as np, requests, soundfile as sf
from pathlib import Path
from IPython.display import Audio, display
logger = logging.getLogger(__name__)

def validate_imports():
    required = [('torch', torch), ('torchaudio', torchaudio), ('numpy', np), ('requests', requests), ('soundfile', sf)]
    for name, module in required:
        if module is None:
            logger.error(f'Module {name} not imported')
            return False
        logger.info(f"{name} version: {getattr(module, '__version__', 'unknown')}")
    try:
        from transformers import AutoModel, AutoProcessor
        import transformers
        logger.info(f'transformers version: {transformers.__version__}')
    except ImportError as e:
        logger.error(f'transformers import failed: {e}')
        return False
    return True

success = validate_imports()
assert success, 'Import validation failed'
print('Cell executed successfully')

In [ ]:
# CELL 4: Download Reference Audio
REFERENCE_AUDIO_URL = 'https://archive.org/download/ppiirqpwrygv7ehggjvlaxcuz6fn8_w/pPiiRQpWrygV7ehgGjvl%2BaXcuz6fn8_w.mp3'
REFERENCE_AUDIO_PATH = Path('/content/reference_audio.mp3')
REFERENCE_AUDIO_CONVERTED = Path('/content/reference_audio.wav')

def download_audio(url, output_path, max_retries=3):
    for attempt in range(max_retries):
        try:
            logger.info(f'Downloading audio (attempt {attempt + 1}/{max_retries})')
            response = requests.get(url, stream=True, timeout=60)
            response.raise_for_status()
            with open(output_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            if output_path.exists() and output_path.stat().st_size > 0:
                logger.info(f'Audio downloaded: {output_path.stat().st_size} bytes')
                return True
        except Exception as e:
            logger.warning(f'Download error: {e}')
        if attempt < max_retries - 1:
            time.sleep(5)
    return False

def convert_audio_format(input_path, output_path):
    try:
        subprocess.run(['ffmpeg', '-y', '-i', str(input_path), '-ar', '16000', '-ac', '1', str(output_path)], check=True, capture_output=True)
        if output_path.exists() and output_path.stat().st_size > 0:
            logger.info('Audio converted successfully')
            return True
    except Exception as e:
        logger.error(f'Conversion error: {e}')
    return False

def validate_audio_file(audio_path):
    try:
        waveform, sample_rate = torchaudio.load(str(audio_path))
        logger.info(f'Audio validated: {sample_rate}Hz, {waveform.shape[1]/sample_rate:.2f}s')
        return True, waveform, sample_rate
    except Exception as e:
        logger.error(f'Audio validation failed: {e}')
        return False, None, None

if not REFERENCE_AUDIO_PATH.exists():
    assert download_audio(REFERENCE_AUDIO_URL, REFERENCE_AUDIO_PATH), 'Audio download failed'
if not REFERENCE_AUDIO_CONVERTED.exists():
    assert convert_audio_format(REFERENCE_AUDIO_PATH, REFERENCE_AUDIO_CONVERTED), 'Audio conversion failed'
valid, waveform, sample_rate = validate_audio_file(REFERENCE_AUDIO_CONVERTED)
assert valid, 'Audio validation failed'
print('Cell executed successfully')

In [ ]:
# CELL 5: Load Qwen3-TTS Model
from transformers import AutoModel, AutoProcessor
import gc

MODEL_NAME = 'Qwen/Qwen3-TTS-0.6B-Instruct'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32

def clear_cuda_cache():
    if DEVICE == 'cuda':
        gc.collect()
        torch.cuda.empty_cache()
        logger.info('CUDA cache cleared')

def load_model(max_retries=3):
    global DTYPE
    for attempt in range(max_retries):
        try:
            logger.info(f'Loading model (attempt {attempt + 1}/{max_retries})')
            clear_cuda_cache()
            processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True, cache_dir='/content/cache')
            logger.info('Processor loaded')
            model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True, torch_dtype=DTYPE, device_map='auto' if DEVICE == 'cuda' else None, cache_dir='/content/cache', low_cpu_mem_usage=True)
            if DEVICE == 'cpu':
                model = model.to(DEVICE)
            model.eval()
            logger.info('Model loaded successfully')
            return model, processor
        except torch.cuda.OutOfMemoryError as e:
            logger.error(f'OOM Error: {e}')
            clear_cuda_cache()
            DTYPE = torch.float32
        except Exception as e:
            logger.error(f'Model loading error: {e}')
        if attempt < max_retries - 1:
            time.sleep(10)
    return None, None

model, processor = load_model()
assert model is not None, 'Model loading failed'
assert processor is not None, 'Processor loading failed'
print('Cell executed successfully')

In [ ]:
# CELL 6: Voice Cloning Configuration
# EDIT YOUR TEXT HERE
input_text = 'Hello, this is a cloned voice generated using Qwen3 TTS on Google Colab.'

voice_cloning_config = {
    'reference_audio': str(REFERENCE_AUDIO_CONVERTED),
    'input_text': input_text,
    'output_path': '/content/generated_output.wav',
    'sample_rate': 16000,
    'temperature': 0.7,
    'top_p': 0.95,
    'top_k': 50,
}

for key, value in voice_cloning_config.items():
    logger.info(f'  {key}: {value}')
print('Cell executed successfully')

In [ ]:
# CELL 7: Generate Speech with Voice Cloning
def generate_speech(model, processor, config, max_retries=3):
    for attempt in range(max_retries):
        try:
            logger.info(f'Generating speech (attempt {attempt + 1}/{max_retries})')
            clear_cuda_cache()
            ref_waveform, ref_sr = torchaudio.load(config['reference_audio'])
            logger.info(f'Reference audio loaded: {ref_waveform.shape}')
            logger.info(f"Input text: {config['input_text']}")
            messages = [{'role': 'user', 'content': [{'type': 'text', 'text': config['input_text']}, {'type': 'audio', 'audio_url': config['reference_audio']}]}]
            text_input = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
            inputs = processor(text=text_input, audios=[config['reference_audio']], return_tensors='pt', sampling_rate=ref_sr)
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            with torch.inference_mode():
                output = model.generate(**inputs, temperature=config.get('temperature', 0.7), top_p=config.get('top_p', 0.95), top_k=config.get('top_k', 50), max_new_tokens=2048, do_sample=True)
            output_path = config['output_path']
            torchaudio.save(output_path, output[0].cpu(), config['sample_rate'])
            if Path(output_path).exists() and Path(output_path).stat().st_size > 0:
                logger.info(f'Speech generated: {Path(output_path).stat().st_size} bytes')
                return True, output_path
        except torch.cuda.OutOfMemoryError as e:
            logger.error(f'OOM Error: {e}')
            clear_cuda_cache()
            config['temperature'] = max(0.5, config.get('temperature', 0.7) - 0.1)
        except Exception as e:
            logger.error(f'Generation error: {e}')
        if attempt < max_retries - 1:
            time.sleep(5)
    return False, None

success, output_path = generate_speech(model, processor, voice_cloning_config)
assert success, 'Speech generation failed'
assert output_path is not None, 'No output path returned'
print('Cell executed successfully')

In [ ]:
# CELL 8: Verify and Play Generated Audio
OUTPUT_PATH = '/content/generated_output.wav'

def verify_and_play_audio(audio_path):
    try:
        audio_file = Path(audio_path)
        if not audio_file.exists():
            logger.error(f'Audio file does not exist: {audio_path}')
            return False
        file_size = audio_file.stat().st_size
        if file_size == 0:
            logger.error(f'Audio file is empty: {audio_path}')
            return False
        logger.info(f'Audio file verified: {file_size} bytes')
        waveform, sample_rate = torchaudio.load(str(audio_path))
        duration = waveform.shape[1] / sample_rate
        logger.info(f'Duration: {duration:.2f} seconds')
        display(Audio(audio_path, rate=sample_rate))
        return True
    except Exception as e:
        logger.error(f'Audio verification failed: {e}')
        return False

success = verify_and_play_audio(OUTPUT_PATH)
assert success, 'Audio verification failed'
print('Cell executed successfully')

In [ ]:
# CELL 9: Final Summary & Cleanup
def print_final_summary():
    logger.info('='*60)
    logger.info('VOICE CLONING COMPLETED SUCCESSFULLY')
    logger.info('='*60)
    output_path = Path('/content/generated_output.wav')
    if output_path.exists():
        logger.info(f'Generated audio: {output_path}')
        logger.info(f'Size: {output_path.stat().st_size} bytes')
        try:
            waveform, sr = torchaudio.load(str(output_path))
            logger.info(f'Duration: {waveform.shape[1]/sr:.2f} seconds')
        except: pass
    logger.info(f'Input Text: {input_text}')
    logger.info(f'Model: {MODEL_NAME}')
    logger.info(f'Device: {DEVICE}')
    logger.info('='*60)
    logger.info('Voice cloning completed successfully')
    logger.info('='*60)

clear_cuda_cache()
print_final_summary()
print('\nAll cells executed successfully!')
print('Voice cloning completed successfully')